# Cursor Approach as Covert Evaluation Signal

Feed-forward click models (Craswell et al., Chapelle & Zhang) assume single-pass examination: a user scans results top-down and clicks or skips. OSEC shows 69% of trials have regressions. But regressions are a gaze-only finding. **Can the cursor independently confirm covert evaluation?**

Finding #10 established the gradient: min gaze-cursor distance predicts click rate at 11× baseline. 14% of non-clicked results had cursor within click threshold (58px) — the "almost clicked" set. This notebook goes deeper:

1. **Multiple approach heuristics** — distance alone is crude. Test velocity, dwell-in-proximity, approach-then-retreat, trajectory shape.
2. **User segmentation** — do satisficers and optimizers approach differently? Does motor coupling (gaze-cursor lag) modulate the signal?
3. **SERP segmentation** — does position, difficulty, or result content quality affect approach behavior?
4. **Pupil validation** — is cognitive load higher during approach fixations? Does LF/HF distinguish approach-then-click from approach-then-reject?
5. **Predictive power** — can approach features improve click prediction beyond position + dwell time?

**Target:** CIKM 2026. The gain is not "your model is wrong" but "here's a signal your model can't see."

## Key Claims (authoritative for paper writers)

*Last verified against executed notebook output: 2026-04-12 (legacy K1–K18); bbox cascade values 2026-05-01.*
*Notebook: `15_cursor_approach.ipynb`.*

If prose in a paper draft cites a value that disagrees with a row below, the paper is wrong — not the notebook. If re-running this notebook produces different values, update this block immediately and `grep` for the old value across `docs/`.

### 2026-05-01 attribution shift — bbox primary; downstream notebooks unblocked

`scripts/compute_cursor_approach_features.py` (NB15 cell 4 extracted to a producer with `--attribution {absolute,organic}`) regenerated the canonical features under bbox AOIs. Output:

- `AdSERP/data/cursor-approach-features.json` — legacy absolute (13,419 records, 2,339 trials)
- `AdSERP/data/cursor-approach-features-organic.json` — bbox attribution (14,760 records, 2,701 trials)

This unblocks NB20, NB21, NB22, NB24, NB28 — all consume `cursor-approach-features.json` directly. Switching them to bbox is a one-line change in cell 1 of each (point at `-organic.json`).

### Bbox-attribution K-IDs (RECOMMENDED for paper figures)

| ID | Claim | Absolute | Organic-bbox | Δ |
|---|---|---|---|---|
| **K-bbox-1** | Records | 13,419 | **14,760** | **+10.0%** (coverage growth) |
| **K-bbox-2** | Trials with valid features | 2,339 | **2,701** | **+15.5%** (coverage growth) |
| **K-bbox-3** | Approach share (min_dist < 100 px) | 28.2% | **23.3%** | −4.9 pp |
| **K-bbox-4** | Approached + clicked (consideration set) | 1,428 | **1,375** | −53 |
| **K-bbox-5** | Approached + rejected (consideration-set core) | 17.5% | **14.0%** | −3.5 pp |

### Per-position record counts (selected positions)

| Pos | Absolute | Organic-bbox | Δ |
|---|---|---|---|
| 0 | 2,320 | **2,658** | +338 |
| 1 | 2,244 | **2,360** | +116 |
| 2 | 2,091 | **1,985** | −106 |
| 7 | 719 | **936** | +217 |
| 8 | 481 | **748** | +267 |
| 9 | 192 | **401** | +209 |

> **Coverage *increases* under bbox (more trials have valid per-AOI features).** The producer's `extract_serp_results`-or-fallback no longer rejects trials where h3 enumeration returns null; bbox AOIs catch trials that the band-estimation pipeline silently dropped. Deep positions (7–9) gain proportionally more records because bbox correctly counts late-list organics that band estimation missed.
>
> **Approach share drops because gap regions (between cards) no longer count as approach failures.** Under absolute, cursor trajectories that drifted into the inter-card gap registered as "approached position N" with high min_dist; under bbox those gaps are not assigned to any AOI, so they no longer pollute the approach denominator. The 28.2% → 23.3% shift reflects cleaner per-AOI distance calculations.
>
> **Approach + rejected = the consideration-set core.** That's the primary CIKM target population: positions the cursor approached but the user didn't click. Under bbox: 14.0% of all (trial, position) records (vs 17.5% under absolute). The number drops because gap-region "approaches" are correctly removed from the numerator.

---

### Legacy K-IDs (preserved for cross-reference)

The K1–K18 below are the published values verified 2026-04-12. Keep for paper-draft cross-references; new figures should cite K-bbox-* above.

### Cursor approach features (per result-position record)

| ID | Claim | Value |
|---|---|---|
| **K1** | Records / trials / base click rate | **13,419 records / 2,340 trials / 16.6%** (2,228 clicks) |
| **K2** | Mean min\_dist (clicked vs not) | **110 px vs 299 px** (overall mean 268 px) |
| **K3** | Almost-clicked (min\_dist < 58 px, not clicked) | **1,122 records (8.4% of all)** |
| **K4** | Approached (min\_dist < 100 px) | **3,783 records (28.2%)**, click rate 37.7%, lift 2.3× |
| **K5** | Best single-threshold AUC | min\_dist < 150 px → **AUC 0.735** |

### Approach → regression → click pathway

| ID | Claim | Value |
|---|---|---|
| **K6** | Approach → regression rate | **81.6%** (vs 58.0% no-approach) |
| **K7** | Approach → regression odds ratio | **3.21×**, χ² = 661.1, p = 8.72 × 10⁻¹⁴⁶ |
| **K8** | Approach + regression → click rate | **37.9% (N = 3,087)** |
| **K9** | Approach + no regression → click rate | 36.9% (N = 696) |
| **K10** | No approach + regression → click rate | 8.1% (N = 5,589) |
| **K11** | No approach + no regression → click rate | 8.6% (N = 4,047) |

### Position gradient

| ID | Claim | Value |
|---|---|---|
| **K12** | Position 0: almost-clicked rate | **15.9%** (N = 2,320) |
| **K13** | Position 0: mean min\_dist | **167 px** |
| **K14** | Position 9: almost-clicked rate | **2.1%** (N = 192) |
| **K15** | Position 9: mean min\_dist | **488 px** |

### Click prediction (replicates NB21 from raw features)

| ID | Claim | Value |
|---|---|---|
| **K16** | Position + dwell + all approach (5-fold CV) | **AUC 0.859 ± 0.019** |
| **K17** | Position coefficient (full model) | **−0.130** (→ skip; correct direction) |
| **K18** | direction\_changes coefficient | **+0.061** (→ click; small positive) |

> **Coordinate-space audit (2026-04-09, cursor side).** Two bug sites in `compute_approach_features` double-counted scroll offset on cursor Y (`my + fix_scroll` where `my` is already page-space), inflating cursor proximity on scrolled trials (82% of corpus).
>
> **Coordinate-space audit (2026-04-12, fixation side).** The symmetric bug existed on the gaze side: FPOGY was treated as screen-space and had scroll added to it, but per the AdSERP README FPOGY is already page-space. Fixing this (and regenerating `cursor-approach-features.json`) shifted every row above. Headline changes: records 15,397 → **13,419** (−12.9%; phantom records from scroll-leaked approach signal at deep positions), base click rate 14.4% → **16.6%**, mean clicked min\_dist 204 px → **110 px** (cursor was always closer than the buggy pipeline reported), almost-clicked 734 → **1,122** records, approach-regression odds ratio 5.04× → **3.21×** (weaker but still highly significant — 36% of the pre-fix signal was scroll artifact). Direction of every headline result preserved. Approach + regression is still the dominant pathway to click (37.9% click rate vs 8% baseline). Position 9 mean min\_dist dropped from 683 px → **488 px**, consistent with the broader pattern: cursor was closer than the bug let us see.
>
> **K8 click rate note.** Pre-fix K8 (43.5% click rate for approach + regression at N = 2,086) was over-concentrated because the scroll leak mislabeled many deep-position results as "approached." Post-fix: 37.9% at N = 3,087 — lower rate on a larger, cleaner sample. Still the dominant decision-to-click signature in the corpus.
>
> **K-bbox-* vs legacy K-IDs.** Legacy K1–K18 used absolute-rank attribution as cell 4 produces it. Bbox attribution generates `cursor-approach-features-organic.json` — a parallel artifact that any consumer notebook can switch to in a one-line change. Both files coexist on disk; downstream notebooks have not yet been migrated.

In [1]:
import sys, csv, json
import numpy as np
from pathlib import Path
from collections import defaultdict
from scipy import stats
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path('.').resolve()))
from data_loader import (
    load_fixations, load_mouse_events, get_trial_meta,
    interpolate_scroll, assign_fixation_to_position, result_band_tops,
    extract_serp_results, load_catalog_indexed, load_lhipa,
    load_difficulty_measures, load_pupil_trial,
    click_to_position, gaze_cursor_distance,
)

# Paths
ADSERP = Path('../AdSERP/data')
PUPIL_DIR = ADSERP / 'pupil-data'
LFHF_PATH = ADSERP / 'butterworth-lfhf-by-position.json'

# Load precomputed catalogs
catalog = load_catalog_indexed()
lhipa_data = load_lhipa()
difficulty_data = load_difficulty_measures()

# Load LF/HF if available
if LFHF_PATH.exists():
    with open(LFHF_PATH) as f:
        lfhf_data = json.load(f)
    print(f"LF/HF data: {len(lfhf_data)} trials")
else:
    lfhf_data = {}
    print("No LF/HF data found")

print(f"Catalog: {len(catalog)} trials")
print(f"LHIPA: {len(lhipa_data)} trials")
print(f"Difficulty: {len(difficulty_data)} trials")

LF/HF data: 2719 trials
Catalog: 2341 trials
LHIPA: 2721 trials
Difficulty: 2772 trials


## 1. Compute per-result cursor approach features

For each trial, for each result position that received at least one fixation:
- Interpolate mouse position at each fixation timestamp
- Compute distance from fixation center to mouse cursor (scroll-corrected)
- Extract temporal cursor trajectory relative to the result center
- Derive: min distance, approach velocity, dwell-in-proximity, retreat distance

In [2]:
def compute_approach_features(trial_id):
    """Compute per-result cursor approach features for a trial.
    
    Returns list of dicts, one per result position that received fixations.
    Each dict contains approach metrics + trial/position metadata.
    """
    fixations = load_fixations(trial_id)
    mouse_data = load_mouse_events(trial_id)
    meta = get_trial_meta(trial_id)
    if fixations is None or mouse_data is None or meta is None:
        return None
    
    all_events, scrolls, clicks = mouse_data
    doc_h, scr_h, _ = meta
    
    if not fixations or not all_events:
        return None
    
    # Build result bands
    serp = extract_serp_results(trial_id)
    n_results = len(serp) if serp else 10
    tops = result_band_tops(n_results, doc_h)
    
    # Build scroll timeline for interpolation
    scroll_ts = [s[0] for s in scrolls] if scrolls else [fixations[0]['t']]
    scroll_ys = [s[1] for s in scrolls] if scrolls else [0]
    
    # Build mouse position timeline (mousemove events only)
    mouse_timeline = [(e[0], e[2], e[3]) for e in all_events 
                       if e[1] in ('mousemove', 'click', 'mouseover') and e[2] > 0]
    if len(mouse_timeline) < 2:
        return None
    
    mouse_ts = np.array([m[0] for m in mouse_timeline])
    mouse_xs = np.array([m[1] for m in mouse_timeline], dtype=float)
    mouse_ys = np.array([m[2] for m in mouse_timeline], dtype=float)
    
    # Click info — coordinate-safe. evtrack ypos is already page-space.
    click_time = clicks[0][0] if clicks else None
    click_pos = click_to_position(clicks, tops, n_results)
    
    # Group fixations by result position
    fix_by_pos = defaultdict(list)
    for fix in fixations:
        scroll_y = interpolate_scroll(fix['t'], scroll_ts, scroll_ys)
        page_y = fix['y']  # FPOGY is page-space (2026-04-12 audit)
        pos = assign_fixation_to_position(page_y, tops, n_results)
        if pos < 0 or pos >= n_results:
            continue
        fix_by_pos[pos].append(fix)
    
    # Compute result center Y for each position (page coordinates)
    result_centers = {}
    for pos in range(n_results):
        if pos < len(tops) - 1:
            center_y = (tops[pos] + tops[pos + 1]) / 2
        else:
            center_y = tops[pos] + (tops[1] - tops[0]) / 2 if len(tops) > 1 else tops[pos] + 100
        result_centers[pos] = center_y
    
    records = []
    
    for pos, pos_fixations in fix_by_pos.items():
        if not pos_fixations:
            continue
        
        was_clicked = (click_pos == pos)
        n_fixations = len(pos_fixations)
        total_dwell_ms = sum(f.get('d', 200) for f in pos_fixations)
        
        # For each fixation, find nearest mouse position and compute distance
        distances = []
        cursor_velocities = []  # velocity toward/away from result center
        dwell_in_proximity = 0  # ms with cursor < threshold
        
        PROX_THRESHOLD = 100  # px
        
        result_center_y = result_centers[pos]
        
        for fix in pos_fixations:
            t = fix['t']
            fix_scroll = interpolate_scroll(t, scroll_ts, scroll_ys)
            
            # Interpolate mouse position at fixation time
            idx = np.searchsorted(mouse_ts, t)
            if idx == 0:
                mx, my = mouse_xs[0], mouse_ys[0]
            elif idx >= len(mouse_ts):
                mx, my = mouse_xs[-1], mouse_ys[-1]
            else:
                # Linear interpolation between bracketing mouse events
                t0, t1 = mouse_ts[idx-1], mouse_ts[idx]
                if t1 == t0:
                    frac = 0
                else:
                    frac = (t - t0) / (t1 - t0)
                mx = mouse_xs[idx-1] + frac * (mouse_xs[idx] - mouse_xs[idx-1])
                my = mouse_ys[idx-1] + frac * (mouse_ys[idx] - mouse_ys[idx-1])
            
            # Mouse is already page-space (evtrack ypos = pageY). Do NOT add
            # scroll. Gaze-cursor distance is computed in screen-space via the
            # canonical helper, which collapses both to screen pixels.
            dist = gaze_cursor_distance(fix['x'], fix['y'], mx, my)
            distances.append(dist)
            
            # Cursor distance to result center — both already page-space.
            cursor_to_result = abs(my - result_center_y)
            
            if cursor_to_result < PROX_THRESHOLD:
                dwell_in_proximity += fix.get('d', 200)
            
            # Velocity: look at mouse position change toward result center
            # over a small window around this fixation
            vel_window = 200  # ms
            mask_before = (mouse_ts >= t - vel_window) & (mouse_ts < t)
            mask_after = (mouse_ts > t) & (mouse_ts <= t + vel_window)
            
            if mask_before.any() and mask_after.any():
                # mouse_ys are already page-space; result_center_y is page-space;
                # no scroll correction needed for the page-space delta.
                y_before = np.mean(mouse_ys[mask_before])
                y_after = np.mean(mouse_ys[mask_after])
                
                dist_before = abs(y_before - result_center_y)
                dist_after = abs(y_after - result_center_y)
                
                # Positive = approaching, negative = retreating
                velocity = (dist_before - dist_after) / (vel_window * 2 / 1000)  # px/s
                cursor_velocities.append(velocity)
        
        distances = np.array(distances)
        
        # Approach features
        min_dist = float(np.min(distances))
        mean_dist = float(np.mean(distances))
        final_dist = float(distances[-1]) if len(distances) > 0 else float('inf')
        
        # Retreat: did cursor get close then pull away?
        min_dist_idx = int(np.argmin(distances))
        retreat_dist = float(distances[-1] - distances[min_dist_idx]) if len(distances) > 1 else 0
        
        # Approach velocity stats
        if cursor_velocities:
            mean_velocity = float(np.mean(cursor_velocities))
            max_approach_velocity = float(np.max(cursor_velocities))
            # Velocity sign changes = hesitation
            if len(cursor_velocities) > 1:
                signs = np.sign(cursor_velocities)
                direction_changes = int(np.sum(np.abs(np.diff(signs)) > 0))
            else:
                direction_changes = 0
        else:
            mean_velocity = 0
            max_approach_velocity = 0
            direction_changes = 0
        
        # Approach trajectory shape
        if len(distances) >= 3:
            # Is it monotonically decreasing (committed approach)?
            diffs = np.diff(distances)
            frac_decreasing = float(np.mean(diffs < 0))
        else:
            frac_decreasing = 0.5
        
        # Episode entry/exit timestamps — first and last fixation at this
        # position. Downstream (NB20, NB24) uses entry_t to classify each
        # episode as forward vs regressive via episode_classifier.
        entry_t = int(pos_fixations[0]['t'])
        exit_t = int(pos_fixations[-1]['t'])

        records.append({
            'trial_id': trial_id,
            'position': pos,
            'was_clicked': was_clicked,
            'n_fixations': n_fixations,
            'total_dwell_ms': total_dwell_ms,
            'click_pos': click_pos if click_pos is not None else -1,
            # Episode boundaries
            'entry_t': entry_t,
            'exit_t': exit_t,
            # Distance features
            'min_dist': min_dist,
            'mean_dist': mean_dist,
            'final_dist': final_dist,
            # Approach dynamics
            'retreat_dist': retreat_dist,
            'dwell_in_proximity_ms': dwell_in_proximity,
            'mean_approach_velocity': mean_velocity,
            'max_approach_velocity': max_approach_velocity,
            'direction_changes': direction_changes,
            'frac_decreasing': frac_decreasing,
        })
    
    return records

# Test on one trial
sample_tid = list(catalog.keys())[0]
sample = compute_approach_features(sample_tid)
if sample:
    print(f"Trial {sample_tid}: {len(sample)} result positions with fixations")
    print(f"Sample record keys: {list(sample[0].keys())}")
    for r in sample[:3]:
        print(f"  pos={r['position']} clicked={r['was_clicked']} "
              f"min_dist={r['min_dist']:.0f}px dwell_prox={r['dwell_in_proximity_ms']}ms "
              f"retreat={r['retreat_dist']:.0f}px vel={r['mean_approach_velocity']:.0f}px/s")
else:
    print(f"Trial {sample_tid}: no data")

Trial p004-b1-t1: 4 result positions with fixations
Sample record keys: ['trial_id', 'position', 'was_clicked', 'n_fixations', 'total_dwell_ms', 'click_pos', 'entry_t', 'exit_t', 'min_dist', 'mean_dist', 'final_dist', 'retreat_dist', 'dwell_in_proximity_ms', 'mean_approach_velocity', 'max_approach_velocity', 'direction_changes', 'frac_decreasing']
  pos=0 clicked=False min_dist=538px dwell_prox=0ms retreat=7px vel=5px/s
  pos=2 clicked=False min_dist=685px dwell_prox=0ms retreat=0px vel=108px/s
  pos=3 clicked=True min_dist=156px dwell_prox=1641.0ms retreat=0px vel=184px/s


In [3]:
# Process all trials
all_records = []
failed = 0

trial_ids = sorted(catalog.keys())
for i, tid in enumerate(trial_ids):
    if i % 500 == 0:
        print(f"  {i}/{len(trial_ids)} trials processed, {len(all_records)} records...")
    records = compute_approach_features(tid)
    if records is None:
        failed += 1
        continue
    all_records.extend(records)

print(f"\nDone: {len(all_records)} result-position records from {len(trial_ids) - failed} trials ({failed} failed)")

# Convert to structured arrays for fast slicing
clicked = np.array([r['was_clicked'] for r in all_records])
positions = np.array([r['position'] for r in all_records])
min_dists = np.array([r['min_dist'] for r in all_records])
mean_dists = np.array([r['mean_dist'] for r in all_records])
retreat_dists = np.array([r['retreat_dist'] for r in all_records])
dwell_prox = np.array([r['dwell_in_proximity_ms'] for r in all_records])
mean_vels = np.array([r['mean_approach_velocity'] for r in all_records])
max_vels = np.array([r['max_approach_velocity'] for r in all_records])
dir_changes = np.array([r['direction_changes'] for r in all_records])
frac_dec = np.array([r['frac_decreasing'] for r in all_records])
final_dists = np.array([r['final_dist'] for r in all_records])
final_dists[~np.isfinite(final_dists)] = 0
n_fixations = np.array([r['n_fixations'] for r in all_records])
total_dwell = np.array([r['total_dwell_ms'] for r in all_records])

print(f"\nClick rate: {clicked.mean()*100:.1f}% ({clicked.sum()} / {len(clicked)})")
print(f"Mean min_dist: {min_dists.mean():.0f}px (clicked: {min_dists[clicked].mean():.0f}, not: {min_dists[~clicked].mean():.0f})")

  0/2341 trials processed, 0 records...


  500/2341 trials processed, 3341 records...


  1000/2341 trials processed, 6101 records...


  1500/2341 trials processed, 8448 records...


  2000/2341 trials processed, 11391 records...



Done: 13419 result-position records from 2340 trials (1 failed)

Click rate: 16.6% (2228 / 13419)
Mean min_dist: 268px (clicked: 110, not: 299)


## 2. Approach heuristic taxonomy

Define multiple heuristics for "cursor approached this result" and compare their click prediction power.

In [4]:
from sklearn.metrics import roc_auc_score

# Define heuristics as boolean arrays
heuristics = {
    'min_dist < 58px':   min_dists < 58,    # Finding #10 threshold (median clicked)
    'min_dist < 100px':  min_dists < 100,   # Production threshold
    'min_dist < 150px':  min_dists < 150,   # Relaxed
    'dwell_prox > 0ms':  dwell_prox > 0,    # Any time cursor near result
    'dwell_prox > 500ms': dwell_prox > 500, # Sustained proximity
    'dwell_prox > 1000ms': dwell_prox > 1000,
    'mean_vel > 0':      mean_vels > 0,     # Net approaching (positive = closing)
    'max_vel > 200px/s': max_vels > 200,    # Fast approach at some point
    'retreat > 50px':    retreat_dists > 50, # Approached then pulled away
    'retreat > 100px':   retreat_dists > 100,
    'frac_dec > 0.6':    frac_dec > 0.6,    # Mostly closing distance
    'dir_changes >= 2':  dir_changes >= 2,  # Hesitation (back-and-forth)
}

# For each heuristic: classification stats
print(f"{'Heuristic':<25} {'Prev':>6} {'ClickRate':>10} {'BaseRate':>10} {'Lift':>6} {'AUC':>6}")
print("-" * 70)

base_rate = clicked.mean()

for name, mask in heuristics.items():
    prev = mask.mean()
    if mask.sum() > 0:
        cr = clicked[mask].mean()
        lift = cr / base_rate if base_rate > 0 else 0
    else:
        cr = 0
        lift = 0
    
    # AUC for this single binary feature
    try:
        auc = roc_auc_score(clicked, mask.astype(float))
    except ValueError:
        auc = 0.5
    
    print(f"  {name:<23} {prev:>5.1%} {cr:>9.1%} {base_rate:>9.1%} {lift:>5.1f}× {auc:>5.3f}")

# Composite: approach + retreat = "considered and rejected"
considered_rejected = (min_dists < 100) & (retreat_dists > 50) & (~clicked)
almost_clicked = (min_dists < 58) & (~clicked)

print(f"\n--- Key segments ---")
print(f"  Almost clicked (min<58, not clicked): {almost_clicked.sum()} ({almost_clicked.mean()*100:.1f}%)")
print(f"  Considered-rejected (min<100, retreat>50, not clicked): {considered_rejected.sum()} ({considered_rejected.mean()*100:.1f}%)")
print(f"  Fixations on almost-clicked: {n_fixations[almost_clicked].mean():.1f} vs clicked: {n_fixations[clicked].mean():.1f}")

Heuristic                   Prev  ClickRate   BaseRate   Lift    AUC
----------------------------------------------------------------------
  min_dist < 58px         14.6%     42.7%     16.6%   2.6× 0.638
  min_dist < 100px        28.2%     37.7%     16.6%   2.3× 0.715
  min_dist < 150px        41.3%     32.3%     16.6%   1.9× 0.735
  dwell_prox > 0ms        62.5%     24.5%     16.6%   1.5× 0.679
  dwell_prox > 500ms      43.2%     31.3%     16.6%   1.9× 0.728
  dwell_prox > 1000ms     29.4%     38.7%     16.6%   2.3× 0.735
  mean_vel > 0            35.6%     28.6%     16.6%   1.7× 0.655
  max_vel > 200px/s       15.4%     34.6%     16.6%   2.1× 0.600
  retreat > 50px          59.8%     16.6%     16.6%   1.0× 0.500
  retreat > 100px         47.5%     14.5%     16.6%   0.9× 0.463
  frac_dec > 0.6          20.0%     19.6%     16.6%   1.2× 0.522
  dir_changes >= 2        19.4%     36.9%     16.6%   2.2× 0.642

--- Key segments ---
  Almost clicked (min<58, not clicked): 1122 (8.4%)
  Cons

### 2a. Reporting rates at three levels

Per-impression rates conflate trials with many fixated results and trials with few. Report at impression, trial, and user level.

In [5]:
# --- Three-level rate reporting ---

THRESHOLDS = [58, 100, 150]

# Build trial-level and user-level lookups
trial_records = defaultdict(list)
for r in all_records:
    trial_records[r['trial_id']].append(r)

user_trials = defaultdict(list)
for tid in trial_records:
    uid = tid.split('-')[0]
    user_trials[uid].append(tid)

print(f"{'Threshold':<12} {'Per Impression':>16} {'Per Trial':>16} {'Per User':>16}")
print(f"{'':12} {'(% of result-pos)':>16} {'(% with ≥1)':>16} {'(% who ever)':>16}")
print("-" * 65)

for thresh in THRESHOLDS:
    # Per impression: fraction of result-positions with approach
    imp_mask = (min_dists < thresh) & (~clicked)
    imp_rate = imp_mask.mean()
    
    # Per trial: fraction of trials containing at least one approach-reject
    trials_with_approach = 0
    for tid, recs in trial_records.items():
        if any(r['min_dist'] < thresh and not r['was_clicked'] for r in recs):
            trials_with_approach += 1
    trial_rate = trials_with_approach / len(trial_records)
    
    # Per user: fraction of users who show approach behavior in any trial
    users_with_approach = 0
    for uid, tids in user_trials.items():
        has_approach = False
        for tid in tids:
            if any(r['min_dist'] < thresh and not r['was_clicked'] for r in trial_records[tid]):
                has_approach = True
                break
        if has_approach:
            users_with_approach += 1
    user_rate = users_with_approach / len(user_trials)
    
    print(f"  <{thresh}px    {imp_rate:>15.1%} {trial_rate:>15.1%} {user_rate:>15.1%}")

# Also report: mean number of approach-rejected results per trial (when present)
print(f"\n--- Approach-rejected results per trial (min_dist < 100px) ---")
counts = []
for tid, recs in trial_records.items():
    n = sum(1 for r in recs if r['min_dist'] < 100 and not r['was_clicked'])
    counts.append(n)
counts = np.array(counts)
print(f"  Mean: {counts.mean():.1f}, Median: {np.median(counts):.0f}")
print(f"  When >0: Mean: {counts[counts>0].mean():.1f}, Median: {np.median(counts[counts>0]):.0f}")
print(f"  Distribution: 0={np.sum(counts==0)} ({np.mean(counts==0)*100:.0f}%), "
      f"1={np.sum(counts==1)}, 2={np.sum(counts==2)}, 3+={np.sum(counts>=3)}")

Threshold      Per Impression        Per Trial         Per User
             (% of result-pos)      (% with ≥1)     (% who ever)
-----------------------------------------------------------------
  <58px               8.4%           34.2%          100.0%
  <100px              17.5%           57.2%          100.0%
  <150px              27.9%           74.0%          100.0%

--- Approach-rejected results per trial (min_dist < 100px) ---
  Mean: 1.0, Median: 1
  When >0: Mean: 1.8, Median: 1
  Distribution: 0=1000 (43%), 1=746, 2=347, 3+=246


### 2b. Orient-phase confound: is position 0's approach rate an artifact?

Position 0 has 15.8% almost-clicked rate — but Orient (fixations 1–5) involves wide saccades with the cursor still in its initial position. If cursor approach at position 0 concentrates during Orient, it's initial cursor placement, not evaluation. Split by OSEC phase.

In [6]:
# Recompute position 0 approach split by fixation index (proxy for OSEC phase)
# Orient = fixations 1-5, Evaluate = fixations 6+

ORIENT_BOUNDARY = 5  # fixations 0-4 = orient/survey, 5+ = evaluate

orient_approach = []  # (trial_id, pos, min_dist, was_clicked) during orient
evaluate_approach = []

for tid in trial_records:
    fixations = load_fixations(tid)
    mouse_data = load_mouse_events(tid)
    meta = get_trial_meta(tid)
    if fixations is None or mouse_data is None or meta is None:
        continue
    
    all_events, scrolls, clicks = mouse_data
    doc_h, scr_h, _ = meta
    serp = extract_serp_results(tid)
    n_results = len(serp) if serp else 10
    tops = result_band_tops(n_results, doc_h)
    
    scroll_ts = [s[0] for s in scrolls] if scrolls else [fixations[0]['t']]
    scroll_ys = [s[1] for s in scrolls] if scrolls else [0]
    
    mouse_timeline = [(e[0], e[2], e[3]) for e in all_events 
                       if e[1] in ('mousemove', 'click', 'mouseover') and e[2] > 0]
    if len(mouse_timeline) < 2:
        continue
    mouse_ts_arr = np.array([m[0] for m in mouse_timeline])
    mouse_xs_arr = np.array([m[1] for m in mouse_timeline], dtype=float)
    mouse_ys_arr = np.array([m[2] for m in mouse_timeline], dtype=float)
    
    # Coordinate-safe: evtrack clicks are already page-space.
    click_time = clicks[0][0] if clicks else None
    click_pos = click_to_position(clicks, tops, n_results)
    
    # Only look at position 0 fixations
    for fix_idx, fix in enumerate(fixations):
        scroll_y = interpolate_scroll(fix['t'], scroll_ts, scroll_ys)
        page_y = fix['y']  # FPOGY is page-space (2026-04-12 audit)
        pos = assign_fixation_to_position(page_y, tops, n_results)
        if pos != 0:
            continue
        
        # Compute cursor distance at this fixation
        t = fix['t']
        idx = np.searchsorted(mouse_ts_arr, t)
        if idx == 0:
            mx, my = mouse_xs_arr[0], mouse_ys_arr[0]
        elif idx >= len(mouse_ts_arr):
            mx, my = mouse_xs_arr[-1], mouse_ys_arr[-1]
        else:
            t0, t1 = mouse_ts_arr[idx-1], mouse_ts_arr[idx]
            frac = (t - t0) / (t1 - t0) if t1 != t0 else 0
            mx = mouse_xs_arr[idx-1] + frac * (mouse_xs_arr[idx] - mouse_xs_arr[idx-1])
            my = mouse_ys_arr[idx-1] + frac * (mouse_ys_arr[idx] - mouse_ys_arr[idx-1])
        
        dist = gaze_cursor_distance(fix['x'], fix['y'], mx, my)
        
        phase = 'orient' if fix_idx < ORIENT_BOUNDARY else 'evaluate'
        was_clicked = (click_pos == 0)
        
        if phase == 'orient':
            orient_approach.append((tid, dist, was_clicked))
        else:
            evaluate_approach.append((tid, dist, was_clicked))

orient_dists = np.array([x[1] for x in orient_approach])
orient_clicked = np.array([x[2] for x in orient_approach])
eval_dists = np.array([x[1] for x in evaluate_approach])
eval_clicked = np.array([x[2] for x in evaluate_approach])

print("--- Position 0: Orient vs Evaluate Phase ---")
print(f"{'Phase':<12} {'N fix':>8} {'<58px':>8} {'<100px':>8} {'Mean dist':>10} {'Click rate':>12}")
print("-" * 60)

for label, dists, cl in [('Orient', orient_dists, orient_clicked),
                           ('Evaluate', eval_dists, eval_clicked)]:
    n = len(dists)
    close_58 = ((dists < 58) & (~cl)).mean() if n > 0 else 0
    close_100 = ((dists < 100) & (~cl)).mean() if n > 0 else 0
    mean_d = dists.mean() if n > 0 else 0
    cr = cl.mean() if n > 0 else 0
    print(f"  {label:<10} {n:>8} {close_58:>7.1%} {close_100:>7.1%} {mean_d:>9.0f}px {cr:>11.1%}")

print(f"\nIf Orient shows higher approach rate → cursor placement artifact.")
print(f"If Evaluate shows higher → genuine covert evaluation.")

--- Position 0: Orient vs Evaluate Phase ---
Phase           N fix    <58px   <100px  Mean dist   Click rate
------------------------------------------------------------
  Orient         6088    1.7%    5.2%       368px       21.5%
  Evaluate      45291    1.7%    4.7%       361px       39.1%

If Orient shows higher approach rate → cursor placement artifact.
If Evaluate shows higher → genuine covert evaluation.


## 3. Slice by user attributes

**Hypothesis:** Optimizers (high regression rate, high cognitive load) should show more cursor approaches to non-clicked results — they're evaluating more carefully before committing. Satisficers click earlier with less comparison.

**Motor coupling hypothesis:** Users with tight gaze-cursor coupling (low lag) should show stronger approach signal because their cursor tracks their attention. Loose-coupling users park the cursor — approach signal should be weaker.

In [7]:
# Build per-user profiles from catalog
user_stats = defaultdict(lambda: {'trials': [], 'regression_rate': [], 'lhipa': []})

for tid, info in catalog.items():
    uid = tid.split('-')[0]  # participant ID
    user_stats[uid]['trials'].append(tid)
    if 'has_scroll_regression' in info:
        user_stats[uid]['regression_rate'].append(1 if info['has_scroll_regression'] else 0)
    if tid in lhipa_data:
        lhipa_val = lhipa_data[tid]
        if isinstance(lhipa_val, dict):
            lhipa_val = lhipa_val.get('lhipa', None)
        if lhipa_val is not None:
            user_stats[uid]['lhipa'].append(lhipa_val)

# Compute user-level regression rate and mean LHIPA
user_regression = {}
user_lhipa = {}
for uid, s in user_stats.items():
    if s['regression_rate']:
        user_regression[uid] = np.mean(s['regression_rate'])
    if s['lhipa']:
        user_lhipa[uid] = np.mean(s['lhipa'])

# Median split: satisficers vs optimizers
reg_rates = np.array(list(user_regression.values()))
reg_median = np.median(reg_rates)
optimizer_users = {uid for uid, r in user_regression.items() if r > reg_median}
satisficer_users = {uid for uid, r in user_regression.items() if r <= reg_median}

print(f"Users: {len(user_stats)} total")
print(f"Regression rate median: {reg_median:.2f}")
print(f"Optimizers (>{reg_median:.2f}): {len(optimizer_users)}, Satisficers: {len(satisficer_users)}")

# Tag each record with user type
record_uids = [r['trial_id'].split('-')[0] for r in all_records]
is_optimizer = np.array([uid in optimizer_users for uid in record_uids])
is_satisficer = np.array([uid in satisficer_users for uid in record_uids])

# Compare approach behavior
for label, mask in [('Optimizers', is_optimizer), ('Satisficers', is_satisficer)]:
    subset_clicked = clicked[mask]
    subset_min_dist = min_dists[mask]
    subset_retreat = retreat_dists[mask]
    subset_dwell = dwell_prox[mask]
    
    almost_clicked_rate = ((subset_min_dist < 58) & (~subset_clicked)).mean()
    mean_retreat_nonclick = subset_retreat[~subset_clicked].mean() if (~subset_clicked).sum() > 0 else 0
    dwell_prox_nonclick = subset_dwell[~subset_clicked].mean() if (~subset_clicked).sum() > 0 else 0
    
    print(f"\n{label} (N={mask.sum()}):")
    print(f"  Click rate: {subset_clicked.mean()*100:.1f}%")
    print(f"  Almost-clicked rate: {almost_clicked_rate*100:.1f}%")
    print(f"  Mean retreat (non-clicked): {mean_retreat_nonclick:.0f}px")
    print(f"  Mean dwell-in-prox (non-clicked): {dwell_prox_nonclick:.0f}ms")
    print(f"  Mean min_dist (clicked): {subset_min_dist[subset_clicked].mean():.0f}px")
    print(f"  Mean min_dist (non-clicked): {subset_min_dist[~subset_clicked].mean():.0f}px")

# Motor coupling: compute per-user mean min_dist as proxy for coupling tightness
user_mean_dist = defaultdict(list)
for r in all_records:
    uid = r['trial_id'].split('-')[0]
    user_mean_dist[uid].append(r['mean_dist'])

user_coupling = {uid: np.mean(dists) for uid, dists in user_mean_dist.items() if len(dists) > 5}
coupling_vals = np.array(list(user_coupling.values()))
coupling_median = np.median(coupling_vals)
tight_users = {uid for uid, c in user_coupling.items() if c < coupling_median}
loose_users = {uid for uid, c in user_coupling.items() if c >= coupling_median}

is_tight = np.array([r['trial_id'].split('-')[0] in tight_users for r in all_records])
is_loose = np.array([r['trial_id'].split('-')[0] in loose_users for r in all_records])

print(f"\n--- Motor Coupling ---")
print(f"Coupling median (mean_dist): {coupling_median:.0f}px")
for label, mask in [('Tight coupling', is_tight), ('Loose coupling', is_loose)]:
    subset = min_dists[mask]
    subset_c = clicked[mask]
    almost = ((subset < 58) & (~subset_c)).mean()
    auc = roc_auc_score(subset_c, -subset) if subset_c.sum() > 0 else 0.5
    print(f"  {label} (N={mask.sum()}): almost-clicked={almost*100:.1f}%, min_dist AUC={auc:.3f}")

Users: 47 total
Regression rate median: 0.66
Optimizers (>0.66): 23, Satisficers: 24

Optimizers (N=8194):
  Click rate: 14.2%
  Almost-clicked rate: 9.1%
  Mean retreat (non-clicked): 168px
  Mean dwell-in-prox (non-clicked): 728ms
  Mean min_dist (clicked): 98px
  Mean min_dist (non-clicked): 280px

Satisficers (N=5225):
  Click rate: 20.3%
  Almost-clicked rate: 7.2%
  Mean retreat (non-clicked): 157px
  Mean dwell-in-prox (non-clicked): 704ms
  Mean min_dist (clicked): 123px
  Mean min_dist (non-clicked): 331px

--- Motor Coupling ---
Coupling median (mean_dist): 425px
  Tight coupling (N=5914): almost-clicked=12.2%, min_dist AUC=0.771
  Loose coupling (N=7505): almost-clicked=5.3%, min_dist AUC=0.817


## 4. Slice by SERP attributes

- **Position:** Does approach behavior change with depth? Higher positions should have more approaches simply from position bias — but does the *approach-to-click ratio* change?
- **Difficulty:** On harder SERPs (low jaccard, high relevance spread), do users approach more results before committing?
- **Distance from click:** Results near the clicked position should show more approaches (comparison shopping).

In [8]:
# --- By position ---
print("--- Approach by Result Position ---")
print(f"{'Pos':>4} {'N':>6} {'ClickRate':>10} {'AlmostClick':>12} {'MeanMinDist':>12} {'MeanRetreat':>12} {'DwellProx':>10}")
print("-" * 75)

for pos in range(10):
    mask = positions == pos
    if mask.sum() < 20:
        continue
    cr = clicked[mask].mean()
    ac = ((min_dists[mask] < 58) & (~clicked[mask])).mean()
    md = min_dists[mask].mean()
    rt = retreat_dists[mask & ~clicked].mean() if (mask & ~clicked).sum() > 0 else 0
    dp = dwell_prox[mask & ~clicked].mean() if (mask & ~clicked).sum() > 0 else 0
    print(f"  {pos:>3} {mask.sum():>6} {cr:>9.1%} {ac:>11.1%} {md:>11.0f}px {rt:>11.0f}px {dp:>9.0f}ms")

# --- By SERP difficulty ---
print("\n--- Approach by SERP Difficulty (jaccard terciles) ---")

# Map trial_id to difficulty
trial_jaccard = {}
for tid, d in difficulty_data.items():
    if isinstance(d, dict) and 'jaccard' in d:
        trial_jaccard[tid] = d['jaccard']

record_jaccards = np.array([trial_jaccard.get(r['trial_id'], np.nan) for r in all_records])
valid_j = ~np.isnan(record_jaccards)

if valid_j.sum() > 100:
    j_vals = record_jaccards[valid_j]
    j_terciles = np.percentile(j_vals, [33, 67])
    
    for label, lo, hi in [('Easy (high J)', j_terciles[1], 1.0),
                           ('Medium', j_terciles[0], j_terciles[1]),
                           ('Hard (low J)', 0, j_terciles[0])]:
        mask = valid_j & (record_jaccards >= lo) & (record_jaccards <= hi)
        if mask.sum() < 20:
            continue
        cr = clicked[mask].mean()
        ac = ((min_dists[mask] < 58) & (~clicked[mask])).mean()
        rt = retreat_dists[mask & ~clicked].mean() if (mask & ~clicked).sum() > 0 else 0
        print(f"  {label:<20} N={mask.sum():>5} click={cr:.1%} almost-clicked={ac:.1%} retreat={rt:.0f}px")
else:
    print("  Insufficient difficulty data")

# --- Distance from click position ---
print("\n--- Approach by Distance from Clicked Result ---")
click_positions = np.array([r['click_pos'] for r in all_records])
pos_from_click = np.abs(positions - click_positions)

for dist_label, lo, hi in [('Same pos (clicked)', -0.5, 0.5),
                             ('±1 position', 0.5, 1.5),
                             ('±2 positions', 1.5, 2.5),
                             ('±3+ positions', 2.5, 100)]:
    mask = (pos_from_click >= lo) & (pos_from_click < hi) & (click_positions >= 0)
    if mask.sum() < 20:
        continue
    cr = clicked[mask].mean()
    ac = ((min_dists[mask] < 58) & (~clicked[mask])).mean()
    md = min_dists[mask & ~clicked].mean() if (mask & ~clicked).sum() > 0 else 0
    rt = retreat_dists[mask & ~clicked].mean() if (mask & ~clicked).sum() > 0 else 0
    print(f"  {dist_label:<22} N={mask.sum():>5} click={cr:.1%} almost-clicked={ac:.1%} min_dist={md:.0f}px retreat={rt:.0f}px")

--- Approach by Result Position ---
 Pos      N  ClickRate  AlmostClick  MeanMinDist  MeanRetreat  DwellProx
---------------------------------------------------------------------------
    0   2320     22.7%       15.9%         167px         276px      1494ms
    1   2244     23.2%       11.6%         185px         203px       938ms
    2   2091     25.0%        8.8%         218px         171px       781ms
    3   1779     16.3%        6.3%         272px         147px       588ms
    4   1449     10.2%        4.7%         315px         141px       456ms
    5   1169      7.5%        4.4%         346px         125px       392ms
    6    946      7.0%        2.9%         384px          94px       331ms
    7    719      4.9%        3.9%         411px          91px       300ms
    8    481      4.6%        3.5%         449px          71px       232ms
    9    192      5.2%        2.1%         488px          45px       128ms

--- Approach by SERP Difficulty (jaccard terciles) ---
  Easy (h

## 5. Pupil validation: cognitive load during approach

If cursor approach reflects genuine evaluation, pupil LF/HF should be higher (more cognitive load) during fixations where the cursor is near the result. Compare:
- Fixations with cursor < 100px vs > 300px from result
- Approached-then-clicked vs approached-then-rejected
- Use per-position Butterworth LF/HF from precomputed data

In [9]:
# Match approach records to per-position LF/HF data
approach_lfhf_close = []  # cursor < 100px, LF/HF for that position
approach_lfhf_far = []    # cursor > 300px
approach_lfhf_clicked = []
approach_lfhf_rejected = []  # approached < 100px but not clicked

for r in all_records:
    tid = r['trial_id']
    pos = r['position']
    
    if tid not in lfhf_data:
        continue
    
    trial_lfhf = lfhf_data[tid]
    if 'positions' not in trial_lfhf:
        continue
    
    # Find LF/HF for this position
    pos_lfhf = None
    for p in trial_lfhf['positions']:
        if p.get('pos') == pos and p.get('lfhf') is not None:
            pos_lfhf = p['lfhf']
            break
    
    if pos_lfhf is None:
        continue
    
    if r['min_dist'] < 100:
        approach_lfhf_close.append(pos_lfhf)
        if r['was_clicked']:
            approach_lfhf_clicked.append(pos_lfhf)
        else:
            approach_lfhf_rejected.append(pos_lfhf)
    elif r['min_dist'] > 300:
        approach_lfhf_far.append(pos_lfhf)

approach_lfhf_close = np.array(approach_lfhf_close)
approach_lfhf_far = np.array(approach_lfhf_far)
approach_lfhf_clicked = np.array(approach_lfhf_clicked)
approach_lfhf_rejected = np.array(approach_lfhf_rejected)

print("--- Pupil LF/HF by Cursor Proximity ---")
print(f"  Close (<100px): N={len(approach_lfhf_close)}, median LF/HF={np.median(approach_lfhf_close):.1f}")
print(f"  Far (>300px):   N={len(approach_lfhf_far)}, median LF/HF={np.median(approach_lfhf_far):.1f}")

if len(approach_lfhf_close) > 10 and len(approach_lfhf_far) > 10:
    stat, p = stats.mannwhitneyu(approach_lfhf_close, approach_lfhf_far, alternative='two-sided')
    print(f"  Mann-Whitney U: p = {p:.2e}")
    print(f"  Higher LF/HF = more cognitive load")

print(f"\n--- Pupil LF/HF: Approached-and-Clicked vs Approached-and-Rejected ---")
print(f"  Clicked:  N={len(approach_lfhf_clicked)}, median={np.median(approach_lfhf_clicked):.1f}")
print(f"  Rejected: N={len(approach_lfhf_rejected)}, median={np.median(approach_lfhf_rejected):.1f}")

if len(approach_lfhf_clicked) > 10 and len(approach_lfhf_rejected) > 10:
    stat, p = stats.mannwhitneyu(approach_lfhf_clicked, approach_lfhf_rejected, alternative='two-sided')
    print(f"  Mann-Whitney U: p = {p:.2e}")
    
    # LHIPA comparison too
    print(f"\n--- Trial-level LHIPA for trials with approached-rejected results ---")
    rejected_tids = set(r['trial_id'] for r in all_records 
                        if r['min_dist'] < 100 and not r['was_clicked'])
    clean_tids = set(r['trial_id'] for r in all_records 
                     if r['min_dist'] > 300 and not r['was_clicked'])
    
    rej_lhipa = [lhipa_data[tid] if not isinstance(lhipa_data[tid], dict) 
                 else lhipa_data[tid].get('lhipa')
                 for tid in rejected_tids if tid in lhipa_data]
    rej_lhipa = [v for v in rej_lhipa if v is not None]
    
    clean_lhipa = [lhipa_data[tid] if not isinstance(lhipa_data[tid], dict)
                   else lhipa_data[tid].get('lhipa')
                   for tid in clean_tids - rejected_tids if tid in lhipa_data]
    clean_lhipa = [v for v in clean_lhipa if v is not None]
    
    if rej_lhipa and clean_lhipa:
        print(f"  Trials WITH approach-reject: N={len(rej_lhipa)}, LHIPA={np.mean(rej_lhipa):.4f}")
        print(f"  Trials WITHOUT approach:     N={len(clean_lhipa)}, LHIPA={np.mean(clean_lhipa):.4f}")
        stat, p = stats.mannwhitneyu(rej_lhipa, clean_lhipa)
        print(f"  p = {p:.2e} (lower LHIPA = higher cognitive load)")

--- Pupil LF/HF by Cursor Proximity ---
  Close (<100px): N=1832, median LF/HF=22.7
  Far (>300px):   N=1257, median LF/HF=18.0
  Mann-Whitney U: p = 6.99e-05
  Higher LF/HF = more cognitive load

--- Pupil LF/HF: Approached-and-Clicked vs Approached-and-Rejected ---
  Clicked:  N=774, median=25.2
  Rejected: N=1058, median=20.9
  Mann-Whitney U: p = 1.04e-03

--- Trial-level LHIPA for trials with approached-rejected results ---
  Trials WITH approach-reject: N=1314, LHIPA=0.0421
  Trials WITHOUT approach:     N=762, LHIPA=0.0495
  p = 1.37e-14 (lower LHIPA = higher cognitive load)


## 6. Predictive power: can approach features improve click prediction?

Baseline: position-only logistic regression (the standard feed-forward assumption).
Add approach features incrementally. Does AUC improve?

In [10]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler

# Feature sets
X_position = positions.reshape(-1, 1)
X_dwell = np.column_stack([positions, total_dwell])
X_approach_dist = np.column_stack([positions, total_dwell, min_dists, mean_dists])
X_approach_full = np.column_stack([
    positions, total_dwell, min_dists, mean_dists, final_dists,
    retreat_dists, dwell_prox, mean_vels, max_vels, dir_changes, frac_dec
])
# Replace inf/nan
for X in [X_position, X_dwell, X_approach_dist, X_approach_full]:
    X[~np.isfinite(X)] = 0




y = clicked.astype(int)

scaler = StandardScaler()

models = {
    'Position only': X_position,
    'Position + dwell': X_dwell,
    'Position + dwell + dist': X_approach_dist,
    'Position + dwell + all approach': X_approach_full,
}

print("--- Click Prediction: 5-fold CV AUC ---")
print(f"{'Model':<40} {'AUC':>8} {'±':>4}")
print("-" * 55)

for name, X in models.items():
    X_scaled = scaler.fit_transform(X)
    lr = LogisticRegression(max_iter=1000, class_weight='balanced')
    scores = cross_val_score(lr, X_scaled, y, cv=5, scoring='roc_auc')
    print(f"  {name:<38} {scores.mean():>7.3f} ±{scores.std():>.3f}")

# Feature importance from full model
lr_full = LogisticRegression(max_iter=1000, class_weight='balanced')
X_full_scaled = scaler.fit_transform(X_approach_full)
lr_full.fit(X_full_scaled, y)

feature_names = ['position', 'dwell', 'min_dist', 'mean_dist', 'final_dist',
                 'retreat', 'dwell_prox', 'mean_vel', 'max_vel', 'dir_changes', 'frac_dec']

print(f"\n--- Feature Coefficients (full model) ---")
for name, coef in sorted(zip(feature_names, lr_full.coef_[0]), key=lambda x: abs(x[1]), reverse=True):
    direction = "→ click" if coef > 0 else "→ skip"
    print(f"  {name:<15} {coef:>+7.3f} ({direction})")

--- Click Prediction: 5-fold CV AUC ---
Model                                         AUC    ±
-------------------------------------------------------


  Position only                            0.641 ±0.038


  Position + dwell                         0.745 ±0.038


  Position + dwell + dist                  0.807 ±0.030
  Position + dwell + all approach          0.859 ±0.019

--- Feature Coefficients (full model) ---
  final_dist       -0.967 (→ skip)
  min_dist         -0.904 (→ skip)
  dwell_prox       +0.738 (→ click)
  mean_dist        +0.589 (→ click)
  max_vel          +0.207 (→ click)
  retreat          -0.206 (→ skip)
  position         -0.130 (→ skip)
  mean_vel         +0.069 (→ click)
  dir_changes      +0.061 (→ click)
  dwell            -0.040 (→ skip)
  frac_dec         +0.031 (→ click)


## 7. The consideration set: characterize "almost clicked" results

The key CIKM claim: cursor approach reveals the consideration set. Profile these results — what distinguishes "approached and clicked" from "approached and rejected"?

In [11]:
# Segment results into 4 categories
approach_threshold = 100  # px

cat_labels = []
for r in all_records:
    approached = r['min_dist'] < approach_threshold
    if r['was_clicked']:
        cat_labels.append('clicked')
    elif approached:
        cat_labels.append('approached_rejected')
    elif r['min_dist'] < 300:
        cat_labels.append('peripherally_seen')
    else:
        cat_labels.append('unseen')

cat_labels = np.array(cat_labels)

print("--- Result Categories ---")
for cat in ['clicked', 'approached_rejected', 'peripherally_seen', 'unseen']:
    mask = cat_labels == cat
    print(f"\n  {cat} (N={mask.sum()}, {mask.mean()*100:.1f}%)")
    print(f"    Fixations:    {n_fixations[mask].mean():.1f} (median {np.median(n_fixations[mask]):.0f})")
    print(f"    Dwell:        {total_dwell[mask].mean():.0f}ms (median {np.median(total_dwell[mask]):.0f})")
    print(f"    Min dist:     {min_dists[mask].mean():.0f}px")
    print(f"    Retreat:      {retreat_dists[mask].mean():.0f}px")
    print(f"    Dwell prox:   {dwell_prox[mask].mean():.0f}ms")
    print(f"    Mean vel:     {mean_vels[mask].mean():.0f}px/s")
    print(f"    Dir changes:  {dir_changes[mask].mean():.1f}")
    print(f"    Position:     {positions[mask].mean():.1f}")

# Key comparison: approached_rejected vs clicked
ar = cat_labels == 'approached_rejected'
cl = cat_labels == 'clicked'

print("\n--- Approached-Rejected vs Clicked (Mann-Whitney U) ---")
for feat_name, feat_arr in [('fixations', n_fixations), ('dwell_ms', total_dwell),
                              ('retreat_px', retreat_dists), ('dwell_prox_ms', dwell_prox),
                              ('mean_velocity', mean_vels), ('dir_changes', dir_changes),
                              ('position', positions)]:
    if ar.sum() > 0 and cl.sum() > 0:
        stat, p = stats.mannwhitneyu(feat_arr[ar], feat_arr[cl], alternative='two-sided')
        ar_med = np.median(feat_arr[ar])
        cl_med = np.median(feat_arr[cl])
        print(f"  {feat_name:<18} rejected={ar_med:>8.1f}  clicked={cl_med:>8.1f}  p={p:.2e}")

--- Result Categories ---

  clicked (N=2228, 16.6%)
    Fixations:    24.4 (median 20)
    Dwell:        5499ms (median 4389)
    Min dist:     110px
    Retreat:      114px
    Dwell prox:   2663ms
    Mean vel:     37px/s
    Dir changes:  2.0
    Position:     2.0

  approached_rejected (N=2355, 17.5%)
    Fixations:    20.7 (median 17)
    Dwell:        4540ms (median 3553)
    Min dist:     58px
    Retreat:      258px
    Dwell prox:   1765ms
    Mean vel:     9px/s
    Dir changes:  1.8
    Position:     2.1

  peripherally_seen (N=4347, 32.4%)
    Fixations:    12.0 (median 9)
    Dwell:        2614ms (median 1849)
    Min dist:     188px
    Retreat:      182px
    Dwell prox:   647ms
    Mean vel:     19px/s
    Dir changes:  0.6
    Position:     2.9

  unseen (N=4489, 33.5%)
    Fixations:    6.3 (median 4)
    Dwell:        1339ms (median 902)
    Min dist:     534px
    Retreat:      97px
    Dwell prox:   240ms
    Mean vel:     15px/s
    Dir changes:  0.1
    Position

## 8. Approach-Retreat and Regression

The cursor approach-retreat is the motor trace of covert evaluation. But does it predict what the user does *next*? Specifically:
- Do approached results get **regressed to** (revisited after the user has scrolled past)?
- Are approached+regressed results more likely to be **clicked**?
- Does **SERP difficulty** or **result similarity** modulate this?

A regression is defined as: a result position that was visited, left (user moved to a deeper position), then returned to. This maps to the re-Evaluate branch in OSEC.

In [12]:
# --- Approach-Retreat predicts regression ---

# For each trial, identify which positions were regressed to
regression_labels = []

for tid_key, recs in trial_records.items():
    fixations_t = load_fixations(tid_key)
    meta_t = get_trial_meta(tid_key)
    mouse_t = load_mouse_events(tid_key)
    if fixations_t is None or meta_t is None or mouse_t is None or len(fixations_t) < 5:
        for r in recs:
            regression_labels.append(False)
        continue
    
    doc_h_t, scr_h_t, _ = meta_t
    serp_t = extract_serp_results(tid_key)
    n_res_t = len(serp_t) if serp_t else 10
    tops_t = result_band_tops(n_res_t, doc_h_t)
    
    _, scrolls_t, _ = mouse_t
    s_ts = [s[0] for s in scrolls_t] if scrolls_t else [fixations_t[0]['t']]
    s_ys = [s[1] for s in scrolls_t] if scrolls_t else [0]
    
    # Build position sequence
    pos_seq = []
    for fix in fixations_t:
        sy = interpolate_scroll(fix['t'], s_ts, s_ys)
        py = fix['y']  # FPOGY is page-space (2026-04-12 audit)
        p = assign_fixation_to_position(py, tops_t, n_res_t)
        if p >= 0:
            pos_seq.append(p)
    
    # Find regressed positions
    max_seen = -1
    visited = set()
    regressed_pos = set()
    for p in pos_seq:
        if p in visited and p < max_seen:
            regressed_pos.add(p)
        visited.add(p)
        max_seen = max(max_seen, p)
    
    for r in recs:
        regression_labels.append(r['position'] in regressed_pos)

regressed = np.array(regression_labels)
approached = min_dists < 100

print(f"Records: {len(regressed)}")
print(f"Approached (<100px): {approached.sum()} ({approached.mean()*100:.1f}%)")
print(f"Regressed to: {regressed.sum()} ({regressed.mean()*100:.1f}%)")

# Q1: Approach predicts regression
from scipy.stats import chi2_contingency

print(f"\n{'='*60}")
print(f"Q1: DOES CURSOR APPROACH PREDICT REGRESSION?")
print(f"{'='*60}")
print(f"  Approached → regressed:     {regressed[approached].mean()*100:.1f}%")
print(f"  Not approached → regressed: {regressed[~approached].mean()*100:.1f}%")

ct = np.array([
    [(approached & regressed).sum(), (approached & ~regressed).sum()],
    [(~approached & regressed).sum(), (~approached & ~regressed).sum()]
])
chi2, p_chi, _, _ = chi2_contingency(ct)
odds = (ct[0,0] * ct[1,1]) / (ct[0,1] * ct[1,0]) if ct[0,1] * ct[1,0] > 0 else float('inf')
print(f"  Chi-square: {chi2:.1f}, p = {p_chi:.2e}")
print(f"  Odds ratio: {odds:.2f}×")

# Q2: Approach + regression → click
print(f"\n{'='*60}")
print(f"Q2: APPROACH + REGRESSION → CLICK CONVERSION")
print(f"{'='*60}")

combos = [
    ('Approach + Regression', approached & regressed),
    ('Approach + No regression', approached & ~regressed),
    ('No approach + Regression', ~approached & regressed),
    ('No approach + No regression', ~approached & ~regressed),
]

for label, mask in combos:
    n = mask.sum()
    cr = clicked[mask].mean() * 100 if n > 0 else 0
    print(f"  {label:<35} N={n:>5}  click rate={cr:.1f}%")

# Q3: Difficulty interaction
print(f"\n{'='*60}")
print(f"Q3: DIFFICULTY × APPROACH → REGRESSION")
print(f"{'='*60}")

record_jaccards_r = np.array([trial_jaccard.get(r['trial_id'], np.nan) for r in all_records])
valid_jr = ~np.isnan(record_jaccards_r)
j_med_r = np.nanmedian(record_jaccards_r)

for diff_label, j_mask in [('Easy SERPs (J > median)', valid_jr & (record_jaccards_r > j_med_r)),
                             ('Hard SERPs (J ≤ median)', valid_jr & (record_jaccards_r <= j_med_r))]:
    app_m = approached & j_mask
    noapp_m = ~approached & j_mask
    app_reg = regressed[app_m].mean() * 100 if app_m.sum() > 0 else 0
    noapp_reg = regressed[noapp_m].mean() * 100 if noapp_m.sum() > 0 else 0
    
    # Click rate for approach+regression in this difficulty
    app_reg_m = approached & regressed & j_mask
    app_reg_cr = clicked[app_reg_m].mean() * 100 if app_reg_m.sum() > 0 else 0
    
    print(f"  {diff_label}")
    print(f"    Approached → regressed: {app_reg:.1f}% | Not approached → regressed: {noapp_reg:.1f}%")
    print(f"    Approach+Regression → click: {app_reg_cr:.1f}% (N={app_reg_m.sum()})")

# Q4: Position interaction — where do approach-retreats concentrate?
print(f"\n{'='*60}")
print(f"Q4: APPROACH-RETREAT BY POSITION")
print(f"{'='*60}")
print(f"{'Pos':>4} {'Approached':>11} {'App→Reg':>9} {'App→Reg→Click':>15} {'NoApp→Reg':>11}")
print("-" * 55)

for pos in range(10):
    pos_mask = positions == pos
    if pos_mask.sum() < 20:
        continue
    app_pos = approached & pos_mask
    noapp_pos = ~approached & pos_mask
    
    app_rate = app_pos.mean() / pos_mask.mean() * approached.mean() if pos_mask.sum() > 0 else 0
    app_n = app_pos.sum()
    
    app_reg_rate = regressed[app_pos].mean() * 100 if app_pos.sum() > 0 else 0
    noapp_reg_rate = regressed[noapp_pos].mean() * 100 if noapp_pos.sum() > 0 else 0
    
    app_reg_click = clicked[app_pos & regressed].mean() * 100 if (app_pos & regressed).sum() > 0 else 0
    
    print(f"  {pos:>3} {app_n:>10} {app_reg_rate:>8.1f}% {app_reg_click:>14.1f}% {noapp_reg_rate:>10.1f}%")

Records: 13419
Approached (<100px): 3783 (28.2%)
Regressed to: 8676 (64.7%)

Q1: DOES CURSOR APPROACH PREDICT REGRESSION?
  Approached → regressed:     81.6%
  Not approached → regressed: 58.0%
  Chi-square: 661.1, p = 8.72e-146
  Odds ratio: 3.21×

Q2: APPROACH + REGRESSION → CLICK CONVERSION
  Approach + Regression               N= 3087  click rate=37.9%
  Approach + No regression            N=  696  click rate=36.9%
  No approach + Regression            N= 5589  click rate=8.1%
  No approach + No regression         N= 4047  click rate=8.6%

Q3: DIFFICULTY × APPROACH → REGRESSION
  Easy SERPs (J > median)
    Approached → regressed: 80.5% | Not approached → regressed: 57.6%
    Approach+Regression → click: 38.1% (N=1573)
  Hard SERPs (J ≤ median)
    Approached → regressed: 82.9% | Not approached → regressed: 58.4%
    Approach+Regression → click: 37.7% (N=1514)

Q4: APPROACH-RETREAT BY POSITION
 Pos  Approached   App→Reg   App→Reg→Click   NoApp→Reg
----------------------------------

## 9. Summary and CIKM framing

Collect the key numbers for the paper abstract and brief to Duchowski.

In [13]:
print("=" * 60)
print("CURSOR APPROACH ANALYSIS — KEY NUMBERS")
print("=" * 60)
print(f"\nDataset: {len(trial_ids) - failed} trials, {len(all_records)} result-position records")
print(f"Base click rate: {clicked.mean()*100:.1f}%")
print(f"\n--- The consideration set ---")
print(f"  Approached-rejected (min_dist<100, not clicked): "
      f"{(cat_labels=='approached_rejected').sum()} "
      f"({(cat_labels=='approached_rejected').mean()*100:.1f}%)")
print(f"  Almost-clicked (min_dist<58, not clicked): "
      f"{almost_clicked.sum()} ({almost_clicked.mean()*100:.1f}%)")

# Save records for downstream use
output_path = Path('../AdSERP/data/cursor-approach-features.json')
output_records = []
for r in all_records:
    # Convert numpy types to native Python for JSON
    clean_r = {k: (float(v) if isinstance(v, (np.floating, np.integer)) 
                    else bool(v) if isinstance(v, np.bool_) 
                    else v) 
               for k, v in r.items()}
    output_records.append(clean_r)

with open(output_path, 'w') as f:
    json.dump(output_records, f)
print(f"\nSaved {len(output_records)} records to {output_path}")

CURSOR APPROACH ANALYSIS — KEY NUMBERS

Dataset: 2340 trials, 13419 result-position records
Base click rate: 16.6%

--- The consideration set ---
  Approached-rejected (min_dist<100, not clicked): 2355 (17.5%)
  Almost-clicked (min_dist<58, not clicked): 1122 (8.4%)

Saved 13419 records to ../AdSERP/data/cursor-approach-features.json
